# EgoLanes

## I. Getting Started Guide

### 1. EgoLanes Model Introduction

EgoLanes is a specialized segmentation network within the VisionPilot suite designed to detect and segment ego-lane
boundaries without relying on HD maps. It follows an Encoder-Context-Decoder pattern, optimized for spatial precision
and global road awareness.

The architecture components include:

- A pretrained EfficientNet-B0 Backbone (Encoder) to extract hierarchical features at five different scales, capturing
everything from fine textures to broad semantic shapes.

- A multi-scale feature fusion layer that downsamples early high-resolution features to a uniform spatial size
($10 \times 20$ in the default configuration), concatenating them into a "deep feature" block.

- Context MLP layers to compress global information and then projects it back into the feature maps using a residual
attention mechanism ($Context \times Features + Features$).

- EgoPath Neck (Decoder) which upsamples the contextualized features back to the target resolution. It uses Skip
Connections from the encoder to re-inject fine-grained spatial data that was lost during downsampling.

- Segmentation Head consisting of 3 convolutional layers produce a 3-channel output, typically representing lane classes
or boundary masks.

As a lane boundary detection model, EgoLanes processes raw image frames and performs real-time semantic segmentation of
driving lanes in the image. It produces a three class segmentation output for the ego-left lane, the ego-right lane and
all other lanes. It outputs lanes at 1/4 resolution of the input image size allowing for quick inference on low power
embedded hardware. EgoLanes was trained with data from a variety of real-world datasets including TuSimple, OpenLane,
CurveLanes, Jiqing, and ONCE3D Lane.

![](../../Media/EgoLanes_GIF_2.gif)

### 2. Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to the next subsection `3. Download Models`.**

First, we need to clone the repository to your local environment. We will use the official repository from the
Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()

# Clone Autoware VisionPilot repo, if not yet done
if not os.path.exists("./autoware_vision_pilot"):
    !git clone https://github.com/autowarefoundation/autoware_vision_pilot.git

The VisionPilot project relies on several key libraries, including **PyTorch** for model execution and
**OpenCV** for image processing.

We will use `pip` to install the required dependencies directly from the `requirements.txt` file provided in 
the `Models` directory.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r autoware_vision_pilot/Models/requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume
you already downloaded them and put them inside directory: 
`autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to Download Pytorch Model Weights (*.pth)](https://drive.google.com/file/d/1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX/view?usp=sharing)
- [Link to Download ONNX FP32 Moel Height (*.onnx)](https://drive.google.com/file/d/1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q/view?usp=sharing)

You can also use below script block, which uses `gdown` to download above model weights into such default directory.

In [ ]:
%pip install gdown

import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX",
        "filepath"  : os.path.join(base_dir, model_dir, "egolanes.pth")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q",
        "filepath"  : os.path.join(base_dir, model_dir, "egolanes.onnx")
    }
}

for weight_type in metadata:

    url         = metadata[weight_type]["url"]
    filepath    = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading EgoLanes {weight_type} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"EgoLanes {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** first.

### 1. Image Inference

The `image_visualization.py` script facilitates batch processing of images by loading a pre-trained model
checkpoint, performing a forward pass on each image, and overlaying the segmented lane lines onto the
original frames.

#### a. Run Batch Image Inference

To run this in any environment, we must navigate to the specific visualization directory to ensure the
script's internal relative paths resolve correctly.

Key Script Parameters:
- `-p / --model_checkpoint_path`: location of your `.pth` file containing the trained weights.
- `-i / --input_image_dirpath`: directory containing `.png`, `.jpg`, or `.jpeg` files you wish to process.
- `-o / --output_image_dirpath`: destination where the script will save the visualization files
(naming them as `[original_id]_data.png`).

In [ ]:
# 1. Path declaration
# Here we use the previously downloaded Pytorch weight file.
# For input and output directories, you can change them to your preferred paths.
# For now we will use the default input and output directories provided in the repository.
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
)
OUTPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/egolanes/images/"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/EgoLanes && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running the inference, we can use matplotlib and PIL to visualize the results directly within this notebook.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Path to the newly created results
result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR) 
    if f.endswith(".png")
])

if (result_files):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3, 
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files)):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we will apply the EgoLanes model to video streams.
The `video_visualization.py` script processes video files frame-by-frame, performs lane segmentation,
and compiles the results into a new annotated video file.

The video inference pipeline utilizes `OpenCV` for frame extraction and tqdm to provide a real-time
progress bar during processing. It leverages the same `make_visualization` logic used in the image
inference section to ensure consistent visual output.

Key script parameters:

- `-p / --model_checkpoint_path`: path to the trained EgoLanes weights.
- `-i / --input_video_filepath`: path to the source video file.
- `-o / --output_video_path`: destination path for the visualized video.
- `-v / --vis`: optional flag to enable a pop-up window showing the frames as they are processed.

Technical details:

- The script automatically resizes input frames to $(640, 320)$ for inference to match the model's requirements, then upscales the result back to $(720, 360)$ for the final video output.
- Release of video pointers and writers are automatically handled upon completion or error.

In [ ]:
# 1. Path declaration
# Similarly as image inference above:
# - Pytorch weight file.
# - Input/Output paths can be changed to preferred.
# - Using defaults for now.
INPUT_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
)
OUTPUT_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/egolanes/videos/highway_normal_output.avi"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/EgoLanes && \
python3 video_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_PATH)} \
    -o {os.path.abspath(OUTPUT_PATH)}

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained model or load vanilla un-trained model for training from scratch

In this repository, model loading behavior is controlled in `Models/training/train_ego_lanes.py`,
then passed into `EgoLanesTrainer(checkpoint_path=...)`.

- If `CHECKPOINT_PATH = None`, trainer initializes a vanilla model (`EgoLanesNetwork`) and trains from scratch.
- If `CHECKPOINT_PATH` points to a `.pth` file, trainer loads weights before training continues.

Use this pattern:

```python
# In Models/training/train_ego_lanes.py
CHECKPOINT_PATH = None  # train from scratch

# OR
CHECKPOINT_PATH = "/absolute/path/to/your_checkpoint.pth"  # resume / fine-tune
```

Then the script already handles both cases:

```python
if (CHECKPOINT_PATH != None):
    trainer = EgoLanesTrainer(checkpoint_path = CHECKPOINT_PATH)
else:
    trainer = EgoLanesTrainer()
```

Practical recommendation:
- Start from scratch when your datasets or labels differ significantly.
- Start from checkpoint when you want faster convergence and stable early performance.

#### b. Prepare the 5 training datasets

EgoLanes training data should be generated with scripts under `Models/data_parsing/EgoLanes/`.

Each parser converts raw annotations into a unified format with this output structure:

```json
--output_dir
    |----image
    |----mask
    |----visualization
    |----drivable_path.json
```

Most scripts **purge `--output_dir` if it already exists**. Use a fresh output path.

From repository root (`autoware_vision_pilot/`), use the following commands.
You can set `--early_stopping` for quick smoke tests before full dataset conversion.

##### i) TuSimple

Expected raw layout (official/Kaggle): includes `train_set/`, `test_set/`, and `test_label.json`.

```bash
python3 Models/data_parsing/EgoLanes/TuSimple/process_tusimple.py \
  --dataset_dir /path/to/TuSimple \
  --output_dir /path/to/processed/TUSIMPLE
```

Optional debug run: add `--early_stopping 100`.

Script outputs: normalized ego lanes + other lanes in `drivable_path.json`, plus 3-channel lane masks in `mask/`.

---

##### ii) CurveLanes

Expected raw layout: `/path/to/CurveLanes/Curvelanes/{train,valid}/...` (the script internally expects top folder name `Curvelanes`).

```bash
python3 Models/data_parsing/EgoLanes/CurveLanes/process_curvelanes.py \
  --dataset_dir /path/to/CurveLanes \
  --output_dir /path/to/processed/CURVELANES \
  --sampling_step 5
```

Notes from parser behavior:
- Auto-resizes/crops mixed source resolutions into a unified training size.
- Interpolates sparse lane polylines.
- Writes lane masks and normalized ego lanes to `drivable_path.json`.

---

##### iii) OpenLane

Expected raw layout includes:
- `images/training`, `images/validation`,
- `lane3d_1000_training/training/...`,
- `lane3d_1000_validation_test/validation/...`.

```bash
python3 Models/data_parsing/EgoLanes/OpenLane/process_openlane.py \
  --dataset_dir /path/to/OpenLane \
  --output_dir /path/to/processed/OPENLANE
```

Notes from parser behavior:
- Crops top 320 px (from $1920\times1280$ to $1920\times960$).
- Filters noisy/insufficient lanes.
- Generates 3-channel masks and normalized lane geometry.

---

##### iv) Jiqing (Jinan-Qingdao Expressway)

Expected raw layout:
- Videos in one folder (e.g., `IMG_0249.MOV`, `IMG_0250.MOV`, ...).
- Matching GT folders in another folder (e.g., `0249/`, `0250/`, each containing frame `.txt` lane files).

```bash
python3 Models/data_parsing/EgoLanes/Jiqing/process_jiqing.py \
  --video_dir /path/to/Jiqing_Videos \
  --gt_dir /path/to/Lane_Parameters \
  --output_dir /path/to/processed/JIQING
```

Notes from parser behavior:
- Reads each video frame with synchronized GT text files.
- Crops bottom 120 px.
- Produces normalized lane JSON + segmentation masks.

---

##### v) Once3DLane

Expected raw layout:
- `images/<segment>/cam01/*.jpg`,
- `lanes/<segment>/cam01/*.json`,
- `infos/<segment>/<segment>.json`.

```bash
python3 Models/data_parsing/EgoLanes/Once3DLane/process_once3d.py \
  --dataset_dir /path/to/Once3DLane \
  --output_dir /path/to/processed/ONCE3DLANE
```

Notes from parser behavior:
- Projects 3D lanes to 2D using camera calibration from each label file.
- Performs lane quality sanity checks.
- Creates masks + normalized ego/other lane labels.

---

**Recommended workflow:**
1. Run each parser first with `--early_stopping` (e.g., 100) to validate paths.
2. Inspect `visualization/` overlays and confirm lane-mask alignment.
3. Re-run full conversion without early stopping.
4. Keep one processed output folder per dataset (e.g., `processed/TUSIMPLE`, `processed/CURVELANES`, ...).

In the next subsection, we will load these processed folders into the EgoLanes training pipeline.

#### c. How to load dataset

`train_ego_lanes.py` loads processed datasets using `LoadDataEgoLanes` from:

```json
--output_dir
    |----image
    |----mask
    |----visualization
    |----drivable_path.json
```

Expected processed tree (for each dataset):

```json
<ROOT_PATH>/
-- TUSIMPLE/
    |----image
    |----mask
    |----visualization
    |----drivable_path.json
-- CURVELANES/
    |----image
    |----mask
    |----visualization
    |----drivable_path.json
-- OPENLANE/
    |----image
    |----mask
    |----visualization
    |----drivable_path.json
-- JIQING/
    |----image
    |----mask
    |----visualization
    |----drivable_path.json
-- ONCE3DLANE/
    |----image
    |----mask
    |----visualization
    |----drivable_path.json
```

In current code, active training datasets are defined by `VALID_DATASET_LIST` in `Models/data_utils/load_data_ego_lanes.py`.

At the moment it is limited to `CURVELANES` and `TUSIMPLE` (others are commented out).

Set data roots in `Models/training/train_ego_lanes.py`:

```python
ROOT_PATH = "/absolute/path/to/processed_datasets_root"
POV_PATH  = "/absolute/path/to/autoware_vision_pilot"
```

The script auto-builds per-dataset paths via:

```python
"path_labels"            : os.path.join(ROOT_PATH, dataset, "drivable_path.json")
"path_masks"             : os.path.join(ROOT_PATH, dataset, "mask")
"path_perspective_image" : os.path.join(ROOT_PATH, dataset, "image")
"path_perspective_vis"   : os.path.join(ROOT_PATH, dataset, "visualization")
```

Before full training, run a quick sanity check:
1. Ensure frame IDs in `drivable_path.json` match image file names in `image/` (e.g., `000123`).
2. Confirm mask count equals image count in each dataset.
3. Inspect several overlay images in `visualization/` to verify lane-mask alignment.

#### d. How to run training

After dataset preparation is complete and paths are configured, training is launched with `train_ego_lanes.py`.

##### i) Step 1: Configure training script

Edit these values in `Models/training/train_ego_lanes.py`:
- `ROOT_PATH`
- `POV_PATH`
- `CHECKPOINT_PATH` (`None` for vanilla model training)
- Training hyperparameters such as `NUM_EPOCHS`, `batch_size`, and logging intervals (`LOGSTEP_LOSS`, `LOGSTEP_VIS`, `LOGSTEP_MODEL`)

If you are unsure about training hyperparameters, leave them as they are.

##### ii) Step 2: Run training

From repository root:

```bash
python3 Models/training/train_ego_lanes.py
```

##### iii) Step 3: What the script does during training

- Builds multi-dataset loaders from `VALID_DATASET_LIST`.
- Applies augmentation through `Augmentations(data_type = "SEGMENTATION")`.
- Runs forward + segmentation losses (BCE + multi-scale edge loss) in `EgoLanesTrainer`.
- Uses gradient accumulation to simulate batch updates (`optimizer.step()` every `batch_size` samples).
- Periodically:
  - Logs training loss to TensorBoard,
  - Logs training visualizations to TensorBoard,
  - Saves checkpoint `.pth` files,
  - Runs validation and logs per-dataset + overall validation losses.

##### **Default save locations in current script**

- Model checkpoints: `Models/saves/AutoSteer/models/`
- Validation visualization figures: `Models/saves/AutoSteer/figures/`

Tip: start with smaller `NUM_EPOCHS` and a larger logging frequency for first debug runs.

#### e. How to visualize training results

This training pipeline provides two main visualization channels:

1. **TensorBoard logs** (loss curves + train visualization panels)
2. **Saved validation figures** (`*_seg.png`, `*_seg_raw.png`) in the figures output directory

![Example of Overall Validation (Tensorboard)](./assets/overall_val_tensorboard_example.png)

##### a) Open TensorBoard

Run in a terminal from project root:

```bash
tensorboard --logdir runs --port 6006
```

Then open `http://localhost:6006` in your browser.

What you should see:
- Scalar curves: training loss, per-dataset validation loss, overall validation loss.
- Image panels: predicted vs ground-truth segmentation overlays logged during training.

##### b) Inspect saved validation images

During periodic validation, the trainer writes files to:
- `Models/saves/AutoSteer/figures/`

Filenames are generated with epoch, step, dataset, and frame id, for example:
- `epoch_2_step_30000_image_TUSIMPLE_000123_seg.png`
- `epoch_2_step_30000_image_TUSIMPLE_000123_seg_raw.png`

`_seg.png` is overlay view; `_seg_raw.png` is raw mask-color view.

If you need more frequent visual feedback during experiments, reduce `LOGSTEP_VIS` and `LOGSTEP_MODEL`.

### 2. Building an Inference Pipeline

#### a. Using ROS2 to publish an image, run inference and visualize results

TBD.

#### b. Using IceOryx to publish an image, run inference and visualize results

TBD.

#### c. Using Zenoh to publish an image, run inference and visualize results

TBD.

#### d. Using Gstreamer to get an image, run inference and visualize results

TBD.